# Filtro de Kalman e Inferencia de Variables Ocultas

**Semana 13 — Taller 1**

Este trabajo aborda la estimación de una variable que no podemos medir de forma directa
(la *variable oculta*) a partir de mediciones que llegan contaminadas con ruido. El filtro
de Kalman resuelve justamente ese problema: combina lo que el modelo predice con lo que el
sensor mide, y pondera ambas fuentes según cuánta confianza merece cada una.

Lo organizamos en dos partes. Primero el caso 1D, que es el más sencillo para ver la
mecánica del filtro paso a paso. Después el caso 2D, donde seguimos un objeto en movimiento
estimando además su velocidad, que nunca se mide.

### Contenido
1. Idea general del filtro
2. Caso 1D: posición de un objeto sobre una recta
3. Análisis del error 1D
4. Caso 2D: seguimiento de un objeto en el plano
5. Sensibilidad a los parámetros de ruido
6. Conclusiones

## 1. Idea general del filtro

El filtro de Kalman trabaja en dos pasos que se repiten en cada instante de tiempo:

**Predicción.** A partir de la estimación anterior, el modelo dice dónde *debería* estar el
sistema ahora. Como ningún modelo es perfecto, la incertidumbre crece un poco en este paso
(la sumamos con $Q$, el ruido del proceso).

**Corrección.** Llega una medición nueva. El filtro compara lo que esperaba con lo que midió
y corrige la estimación. Cuánto corrige depende de la *ganancia de Kalman* $K$: si el sensor
es muy ruidoso ($R$ grande), $K$ es pequeña y casi no le hacemos caso a la medida; si el
sensor es confiable, $K$ crece y la medición pesa más.

En la versión escalar (1D) las ecuaciones quedan así:

$$\hat{x}^- = \hat{x}_{k-1} \qquad P^- = P_{k-1} + Q$$
$$K = \frac{P^-}{P^- + R}$$
$$\hat{x}_k = \hat{x}^- + K\,(z_k - \hat{x}^-) \qquad P_k = (1-K)\,P^-$$

donde $z_k$ es la medición ruidosa y $P$ representa la incertidumbre de la estimación.

> Nota sobre las librerías: usamos `numpy.random.default_rng()`, el generador recomendado
> en NumPy 2.x. Las funciones globales tipo `np.random.randn` siguen existiendo pero quedaron
> como API heredada, así que preferimos la nueva para que el código no dependa de un estado
> global y los resultados sean reproducibles con una semilla.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Un único generador con semilla fija: así el experimento se repite igual cada vez.
rng = np.random.default_rng(42)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

## 2. Caso 1D: posición de un objeto sobre una recta

Simulamos un objeto que se desplaza siguiendo un camino aleatorio (cada paso se suma al
anterior). Esa es la posición *real*, la que en un problema verdadero no conoceríamos. El
sensor nos entrega esa posición pero con ruido gaussiano encima, que es lo único que el
filtro podrá ver.

In [ ]:
n = 50

# Posición real: camino aleatorio acumulado. Es la variable oculta que queremos recuperar.
real = np.cumsum(rng.standard_normal(n))

# Medición: la posición real más ruido del sensor (desviación estándar de 2 unidades).
sensor_std = 2.0
observed = real + rng.normal(0, sensor_std, size=n)

print(f"Muestras generadas: {n}")
print(f"Rango de la posición real: [{real.min():.2f}, {real.max():.2f}]")

Implementamos el filtro como una función para poder reutilizarlo después con distintos
parámetros. Recorre las mediciones una a una y devuelve, además de la estimación, la
incertidumbre $P$ y la ganancia $K$ en cada paso, que nos sirven para analizar el
comportamiento.

In [ ]:
def filtro_kalman_1d(mediciones, Q=0.001, R=4.0, x_inicial=0.0, P_inicial=1.0):
    """Filtro de Kalman escalar.

    mediciones : secuencia de valores observados (ruidosos)
    Q          : varianza del ruido del proceso
    R          : varianza del ruido de medición
    Devuelve la estimación, la incertidumbre P y la ganancia K en cada instante.
    """
    x_hat = x_inicial
    P = P_inicial

    estimacion = np.empty(len(mediciones))
    incertidumbre = np.empty(len(mediciones))
    ganancia = np.empty(len(mediciones))

    for i, z in enumerate(mediciones):
        # Predicción
        x_prior = x_hat
        P_prior = P + Q

        # Corrección
        K = P_prior / (P_prior + R)
        x_hat = x_prior + K * (z - x_prior)
        P = (1 - K) * P_prior

        estimacion[i] = x_hat
        incertidumbre[i] = P
        ganancia[i] = K

    return estimacion, incertidumbre, ganancia


# R coherente con el ruido del sensor: varianza = sensor_std**2 = 4.
# Q se elige acorde con la dinámica del proceso: el camino aleatorio tiene pasos de
# varianza ~1, así que un Q demasiado pequeño (p. ej. 0.001) deja al filtro rígido y
# retrasado. Con Q=0.5 sigue los cambios reales sin amplificar el ruido del sensor.
estimacion, P_hist, K_hist = filtro_kalman_1d(observed, Q=0.5, R=4.0)
print("Estimación calculada para las", len(estimacion), "muestras.")

### Comparación de las tres señales

Graficamos juntas la posición real, la medición ruidosa y la estimación del filtro. Se
espera que la línea estimada quede mucho más suave que la medición y pegada a la trayectoria
real.

In [ ]:
plt.figure()
plt.plot(real, label="Real (oculta)", color="black", linewidth=2)
plt.plot(observed, label="Medición ruidosa", color="tab:red",
         alpha=0.5, marker="o", markersize=3, linestyle="--")
plt.plot(estimacion, label="Estimación Kalman", color="tab:blue", linewidth=2)
plt.title("Filtro de Kalman 1D: estimación de la posición oculta")
plt.xlabel("Instante de tiempo")
plt.ylabel("Posición")
plt.legend()
plt.show()

## 3. Análisis del error 1D

Para no quedarnos solo en lo visual, medimos el error cuadrático medio (MSE) de la medición
y de la estimación frente a la posición real. Si el filtro cumple su función, su error debe
ser claramente menor que el del sensor.

In [ ]:
mse_medicion = np.mean((observed - real) ** 2)
mse_estimacion = np.mean((estimacion - real) ** 2)
reduccion = 100 * (1 - mse_estimacion / mse_medicion)

print(f"MSE de la medición   : {mse_medicion:.3f}")
print(f"MSE de la estimación : {mse_estimacion:.3f}")
print(f"Reducción del error  : {reduccion:.1f}%")

También conviene mirar cómo evolucionan la incertidumbre $P$ y la ganancia $K$. Ambas
arrancan altas y se estabilizan rápido: el filtro empieza dudando y, conforme acumula
mediciones, llega a un régimen estable donde confía una cantidad fija en cada nueva medida.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(P_hist, color="tab:green")
ax1.set_title("Incertidumbre P")
ax1.set_xlabel("Instante de tiempo")
ax1.set_ylabel("P")

ax2.plot(K_hist, color="tab:purple")
ax2.set_title("Ganancia de Kalman K")
ax2.set_xlabel("Instante de tiempo")
ax2.set_ylabel("K")

plt.tight_layout()
plt.show()

## 4. Caso 2D: seguimiento de un objeto en el plano

Ahora el problema interesante. Seguimos un objeto que se mueve en un plano y solo medimos su
posición $(x, y)$. La velocidad nunca se observa, pero el filtro la estima como parte del
estado: ese es el verdadero ejemplo de inferencia de una variable oculta.

El estado tiene cuatro componentes:

$$\mathbf{x} = [x,\; y,\; v_x,\; v_y]^T$$

y usamos un modelo de velocidad constante. Las matrices son:

- $F$: transición de estado (la posición avanza según la velocidad; $dt$ es el paso de tiempo).
- $H$: observación (de las cuatro variables solo medimos las dos posiciones).
- $Q$: ruido del proceso (qué tanto puede desviarse el movimiento real del modelo).
- $R$: ruido de medición (qué tan ruidoso es el sensor de posición).

In [ ]:
dt = 1.0

# Modelo de velocidad constante: x_nuevo = x + vx*dt, y_nuevo = y + vy*dt
F = np.array([
    [1, 0, dt, 0],
    [0, 1, 0, dt],
    [0, 0, 1,  0],
    [0, 0, 0,  1],
], dtype=float)

# Solo medimos posición, no velocidad.
H = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
], dtype=float)

Q = 0.01 * np.eye(4)            # ruido del proceso
R = np.diag([9.0, 9.0])         # ruido de medición (sensor con desv. estándar 3 en cada eje)

print("F =\n", F)
print("H =\n", H)

### Generación de la trayectoria y las mediciones

Creamos la trayectoria real con una velocidad inicial y una pequeña perturbación en cada
paso. Luego simulamos el sensor sumando ruido a la posición.

In [ ]:
pasos = 60

estado = np.array([0.0, 0.0, 1.0, 0.5])   # arranca en el origen con velocidad (1.0, 0.5)
trayectoria_real = np.zeros((pasos, 4))
mediciones = np.zeros((pasos, 2))

for t in range(pasos):
    # El objeto avanza según F y sufre una leve perturbación aleatoria.
    estado = F @ estado + rng.normal(0, 0.05, size=4)
    trayectoria_real[t] = estado
    # El sensor entrega la posición con ruido.
    mediciones[t] = H @ estado + rng.multivariate_normal([0, 0], R)

print("Trayectoria y mediciones generadas:", trayectoria_real.shape, mediciones.shape)

### Filtro de Kalman 2D

La estructura es la misma que en 1D pero con matrices. La ganancia ahora involucra una
inversión de matriz, y el estado se inicializa con una incertidumbre grande porque al
principio no sabemos casi nada de la posición ni de la velocidad.

In [ ]:
def filtro_kalman_2d(mediciones, F, H, Q, R, x0, P0):
    n = len(mediciones)
    dim = F.shape[0]
    x = x0.copy()
    P = P0.copy()
    I = np.eye(dim)

    estimaciones = np.zeros((n, dim))
    for t in range(n):
        # Predicción
        x = F @ x
        P = F @ P @ F.T + Q

        # Corrección
        y = mediciones[t] - H @ x            # innovación
        S = H @ P @ H.T + R                  # covarianza de la innovación
        K = P @ H.T @ np.linalg.inv(S)       # ganancia de Kalman
        x = x + K @ y
        P = (I - K @ H) @ P

        estimaciones[t] = x
    return estimaciones


x_inicial = np.zeros(4)
P_inicial = np.eye(4) * 500.0   # incertidumbre inicial alta

estim_2d = filtro_kalman_2d(mediciones, F, H, Q, R, x_inicial, P_inicial)
print("Estimación 2D lista:", estim_2d.shape)

### Trayectoria en el plano

Comparamos la trayectoria real, las mediciones ruidosas y la estimación del filtro sobre el
plano $XY$.

In [ ]:
plt.figure(figsize=(8, 8))
plt.plot(trayectoria_real[:, 0], trayectoria_real[:, 1],
         label="Trayectoria real", color="black", linewidth=2)
plt.scatter(mediciones[:, 0], mediciones[:, 1],
            label="Mediciones", color="tab:red", alpha=0.5, s=25)
plt.plot(estim_2d[:, 0], estim_2d[:, 1],
         label="Estimación Kalman", color="tab:blue", linewidth=2)
plt.title("Filtro de Kalman 2D: seguimiento en el plano")
plt.xlabel("Posición X")
plt.ylabel("Posición Y")
plt.legend()
plt.axis("equal")
plt.show()

### La variable verdaderamente oculta: la velocidad

Esto es lo más relevante del caso 2D. Nunca medimos la velocidad, pero el filtro la
reconstruye a partir de cómo cambian las posiciones. Comparamos la velocidad estimada en
cada eje contra la real.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(trayectoria_real[:, 2], label="vx real", color="black", linewidth=2)
ax1.plot(estim_2d[:, 2], label="vx estimada", color="tab:blue")
ax1.set_title("Velocidad en X (variable oculta)")
ax1.set_xlabel("Instante de tiempo")
ax1.set_ylabel("vx")
ax1.legend()

ax2.plot(trayectoria_real[:, 3], label="vy real", color="black", linewidth=2)
ax2.plot(estim_2d[:, 3], label="vy estimada", color="tab:blue")
ax2.set_title("Velocidad en Y (variable oculta)")
ax2.set_xlabel("Instante de tiempo")
ax2.set_ylabel("vy")
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Sensibilidad a los parámetros de ruido

$Q$ y $R$ no son números mágicos: marcan en quién confía el filtro. Reutilizamos el caso 1D
probando tres configuraciones para ver el efecto.

- Configuración equilibrada: $Q$ acorde con la dinámica del proceso y $R$ acorde con el
  sensor. Es la que mejor sigue la señal real.
- $Q$ muy pequeño: el filtro asume que la posición casi no cambia, queda rígido y se retrasa
  frente a la trayectoria real (de hecho puede dar peor error que la propia medición).
- $R$ pequeño: el filtro le cree casi todo al sensor y la estimación se parece a la medición
  ruidosa.

In [ ]:
configuraciones = [
    ("Q=0.5, R=4 (equilibrado)", 0.5, 4.0),
    ("Q=0.001, R=4 (confía de más en el modelo)", 0.001, 4.0),
    ("Q=0.5, R=0.5 (confía de más en el sensor)", 0.5, 0.5),
]

plt.figure()
plt.plot(real, label="Real", color="black", linewidth=2)
plt.plot(observed, label="Medición", color="tab:red", alpha=0.3, linestyle="--")

for etiqueta, q, r in configuraciones:
    est, _, _ = filtro_kalman_1d(observed, Q=q, R=r)
    mse = np.mean((est - real) ** 2)
    plt.plot(est, label=f"{etiqueta} | MSE={mse:.2f}")

plt.title("Efecto de Q y R sobre la estimación")
plt.xlabel("Instante de tiempo")
plt.ylabel("Posición")
plt.legend(fontsize=8)
plt.show()

## 6. Conclusiones

- El filtro de Kalman recupera una señal limpia a partir de mediciones ruidosas combinando
  predicción del modelo y corrección con cada nueva medida. En el caso 1D la reducción del
  error frente a la medición directa fue notable.
- En el caso 2D quedó clara la idea de inferencia de variables ocultas: estimamos la
  velocidad sin medirla nunca, deduciéndola del cambio de posición.
- La elección de $Q$ y $R$ define el comportamiento. No hay un valor universal; depende de
  cuánta confianza merecen el modelo y el sensor en cada aplicación.
- Estas mismas ideas se trasladan a seguimiento de objetos en visión por computador,
  localización en robótica y predicción de series temporales, donde casi siempre se mide
  una parte del sistema y hay que inferir el resto.